# 01 — Scan the 2024 fitter trees: probe counts per eta-R9 map cell

Created 2026-10-03. Class: deliverable (input to `02_map_and_efficiency_study.ipynb`).

**Purpose.** The old (repo, `2024_R9_vs_eta_HggOverDY_presel_vf.root`) and new (`2024Lowmass_R9_vs_eta_ratio_presel.root`)
`withPresel` weights are both a lookup in a 25×16 (R9 × |eta|) map. So for every fitter bin
(leg, set, category, pT bin) the weighted pass / fail yields under **any** map are
`sum over cells of N_pass(cell) × map(cell)`. This notebook counts unweighted probes per map cell once; notebook 02 applies the maps.

**Inputs**
- New trees: `fitter_inputs_2024_newPresel_20261001/fitter_input_2024_{Seed,Unseed}_SS.root` (all branches but the two `withPresel` weights are byte-identical to the old trees, `presel_weight_validation.json`).
- Old trees: `fitter_inputs_2024_HggOverDY/…` — only `r9etaWeight_with_presel` is read, to validate the lookup.
- Settings: the 20 shipped `settings_*_HLT_2024_withPresel[_fullPath].py` (`lxplus_2024_newPresel_20261001/`); cuts, flags and pT bins are parsed from them, not retyped.

**Outputs**: `data/cell_counts.npz`, `data/scan_validation.json`.

```mermaid
flowchart LR
 A[Config] --> B[Parse settings]
 A --> C[Load both maps]
 B --> D[Scan loop over chunks]
 C --> D
 D --> E[Validate]
 E --> F[Save]
```

Validation covers: the per-cell lookup reproduces the stored old and new weights for **every** selected entry; entry counts; per-bin weighted yields recomputed from cell counts equal direct sums of the stored weights.
Not covered: what the fitter extracts from these yields (the fit is not redone here). Mass window for counting is 60–120 GeV (the range shown on the fit plots); the fitter histograms span 50–130 (`tnpEGM_fitter.py:118`).

## [A] Config
`uproot` reads ROOT trees without ROOT; `numpy` does the counting.

In [1]:
# [A] Config
from pathlib import Path
import re, json, time
import numpy as np
import uproot   # ROOT file I/O in pure Python

BASE      = Path('/home/santanu/Pictures/crab_2022_2023_2024')
NEW_TREES = BASE / 'fitter_inputs_2024_newPresel_20261001'
OLD_TREES = BASE / 'fitter_inputs_2024_HggOverDY'
MAPS      = {'old': BASE / 'MC_R9eta_weights/2024/2024_R9_vs_eta_HggOverDY_presel_vf.root',
             'new': BASE / 'MC_R9eta_weights/2024/2024Lowmass_R9_vs_eta_ratio_presel.root'}
SETTINGS  = Path('/home/santanu/Documents/Codex/2026-10-01/fin/outputs/lxplus_2024_newPresel_20261001')
OUT       = Path('/home/santanu/Desktop/HLT_2024_lowmass_final_with_NR_eta_r9_rw/etaR9_map_study/data')
OUT.mkdir(parents=True, exist_ok=True)
TREE      = 'PhotonToRECO/fitter_tree'
CATS      = ['EB_R9_0p50to0p55', 'EB_R9_0p55to0p85', 'EB_R9gt0p85', 'EE_R9_0p90to0p93', 'EE_R9gt0p93']
LEGS      = {'seed': 'Seed', 'unseed': 'Unseed'}
SETS      = {'normal': '', 'fullPath': '_fullPath'}
MASS_WIN  = (60.0, 120.0)
CHUNK     = 4_000_000

## [B] Parse settings
Each settings file is executed as Python (`exec`) with a stub `tnpSample`, so `cutBase`, `flags`, `additionalCuts` and `biningDef` come straight from the files the fitter used. `to_numpy_expr` turns the ROOT `TTreeFormula` syntax into a numpy expression: every `&&` / `||` term is wrapped in parentheses because numpy's `&` binds tighter than `>`.

In [2]:
# [B] Parse settings
def to_numpy_expr(s):
    return '(' + s.replace('&&', ') & (').replace('||', ') | (') + ')'

cfg = {}
for leg in LEGS:
    for st, suf in SETS.items():
        for cat in CATS:
            ns = {}
            src = (SETTINGS / f'settings_{cat}_{leg}_HLT_2024_withPresel{suf}.py').read_text()
            src = src.replace('from libPython.tnpClassUtils import tnpSample', 'tnpSample = lambda *a, **k: k')
            exec(src, ns)
            (flag,) = ns['flags'].values()
            pt_edges = np.array([b for b in ns['biningDef'] if b['var'] == 'ph_sc_et'][0]['bins'])
            add = set(ns['additionalCuts'].values())
            assert len(add) == 1 and len(ns['additionalCuts']) == len(pt_edges) - 1
            assert ns['samplesDef']['data']['weight'] == 'weight_with_presel'
            cfg[(leg, st, cat)] = dict(cut=to_numpy_expr(ns['cutBase']) + ' & ' + to_numpy_expr(add.pop()),
                                       flag=to_numpy_expr(flag), pt=pt_edges,
                                       eta=np.array([b for b in ns['biningDef'] if b['var'] == 'ph_sc_abseta'][0]['bins']))
print(len(cfg), 'settings parsed')
k = ('unseed', 'fullPath', 'EE_R9gt0p93'); print(k, cfg[k]['cut'], '| pass:', cfg[k]['flag'], '| pT:', cfg[k]['pt'])
IDENT = set()
for c in cfg.values():
    IDENT |= set(re.findall(r'[A-Za-z_]\w*', c['cut'] + ' ' + c['flag'])) - {'abs'}
BRANCHES = sorted(IDENT | {'ph_sc_et', 'ph_sc_abseta', 'pair_mass', 'probe_full5x5_r9', 'probe_eta', 'r9etaWeight_with_presel'})
print(len(BRANCHES), 'branches:', BRANCHES)

20 settings parsed
('unseed', 'fullPath', 'EE_R9gt0p93') (tag_Ele_pt > 35 ) & ( abs(tag_sc_eta) < 2.5 ) & ( tag_mva > -0.6 ) & ( probe_pt > 18 ) & ( ph_sc_abseta > 1.566 ) & ( ph_sc_abseta < 2.5 ) & ( probe_mva > -0.9 ) & ( probe_hoe < 0.08 ) & ( ph_full5x5x_r9 > 0.93 ) & ( probe_sieie < 0.032 ) & ( probe_phoIso_rhoCorr < 3.0 ) & ( probe_trkSumPtHollowConeDR03 < 6.0 ) & ( tag_seededPass == 1 ) & ( (ph_full5x5x_r9 > 0.8 ) | ( probe_pfRelIso03_chg_quad*probe_pt < 20 ) | ( probe_pfRelIso03_chg_quad < 0.3) ) & ( probe_passFullPath == 1) & (ph_full5x5x_r9 > 0.93) | pass: ((probe_passUnseed == 1)) | pT: [0.00000e+00 2.10000e+01 2.50000e+01 2.80000e+01 3.10000e+01 3.50000e+01
 4.00000e+01 4.50000e+01 5.00000e+01 6.00000e+01 7.00000e+01 9.00000e+01
 9.99999e+05]
21 branches: ['pair_mass', 'ph_full5x5x_r9', 'ph_sc_abseta', 'ph_sc_et', 'probe_eta', 'probe_full5x5_r9', 'probe_hoe', 'probe_mva', 'probe_passFullPath', 'probe_passSeed', 'probe_passUnseed', 'probe_pfRelIso03_chg_quad', 'probe_phoIso_

## [C] Load both maps
`to_numpy` returns cell contents and edges. Both maps must share one binning, otherwise the per-cell method does not apply.

In [3]:
# [C] Load both maps
maps = {}
for k, p in MAPS.items():
    with uproot.open(p) as f:
        v, xe, ye = f['R9_vs_eta'].to_numpy()
    maps[k] = v
    if k == 'old':
        R9E, ETAE = xe, ye
    else:
        assert np.array_equal(xe, R9E) and np.array_equal(ye, ETAE)
NX, NY = len(R9E) - 1, len(ETAE) - 1
print('binning', NX, 'x', NY, '| R9 edges', R9E[[0, -1]], '| |eta| edges', ETAE[[0, -1]])

def cell_index(r9, eta):
    # same convention as replace_2024_presel_weights.ipynb: full5x5 R9, |probe_eta|, clamp to the edge cells
    ix = np.clip(np.digitize(r9, R9E) - 1, 0, NX - 1)
    iy = np.clip(np.digitize(np.abs(eta), ETAE) - 1, 0, NY - 1)
    return ix * NY + iy

binning 25 x 16 | R9 edges [0. 2.] | |eta| edges [0.  2.5]


## [D] Scan loop over chunks
For each leg, the new tree and the old tree are read in lockstep (`iterate` with the same `step_size`). In each chunk and fitter bin, `np.bincount` histograms the selected probes by (pT bin, pass/fail, map cell). Running checks: looked-up map values must equal the stored weights in both trees.

In [4]:
# [D] Scan loop over chunks
NPT = 12
counts = {key: np.zeros((NPT, 2, NX * NY), dtype=np.int64) for key in cfg}         # [pt, fail/pass, cell]
wsum   = {key: np.zeros((NPT, 2, 2), dtype=np.float64) for key in cfg}             # [pt, fail/pass, old/new] direct sums of stored weights
check  = dict(entries={}, mismatch_old={}, mismatch_new={}, selected={})
mask_flat = {}
for leg, LEG in LEGS.items():
    t0 = time.time(); n = mis_o = mis_n = 0
    new_it = uproot.open(NEW_TREES / f'fitter_input_2024_{LEG}_SS.root')[TREE].iterate(BRANCHES, step_size=CHUNK, library='np')
    old_it = uproot.open(OLD_TREES / f'fitter_input_2024_{LEG}_SS.root')[TREE].iterate(['r9etaWeight_with_presel', 'event', 'probe_eta'], step_size=CHUNK, library='np')
    for a, o in zip(new_it, old_it):
        m = len(a['ph_sc_et']); n += m
        assert len(o['event']) == m and np.array_equal(o['probe_eta'], a['probe_eta'])
        cell = cell_index(a['probe_full5x5_r9'], a['probe_eta'])
        win = (a['pair_mass'] > MASS_WIN[0]) & (a['pair_mass'] < MASS_WIN[1])
        ns = dict(a); ns['abs'] = np.abs
        for st in SETS:
            for cat in CATS:
                c = cfg[(leg, st, cat)]
                sel = eval(c['cut'], {}, ns) & win & (a['ph_sc_abseta'] >= c['eta'][0]) & (a['ph_sc_abseta'] < c['eta'][-1])
                ipt = np.digitize(a['ph_sc_et'], c['pt']) - 1
                sel &= (ipt >= 0) & (ipt < len(c['pt']) - 1)
                p = eval(c['flag'], {}, ns).astype(np.int64)
                s_cell, s_pt, s_p = cell[sel], ipt[sel], p[sel]
                wo, wn = o['r9etaWeight_with_presel'][sel], a['r9etaWeight_with_presel'][sel]
                mis_o += int((maps['old'].ravel()[s_cell].astype(np.float32) != wo).sum())
                mis_n += int((maps['new'].ravel()[s_cell].astype(np.float32) != wn).sum())
                flat = (s_pt * 2 + s_p) * (NX * NY) + s_cell
                counts[(leg, st, cat)] += np.bincount(flat, minlength=NPT * 2 * NX * NY).reshape(NPT, 2, NX * NY)
                idx = s_pt * 2 + s_p
                wsum[(leg, st, cat)][..., 0] += np.bincount(idx, weights=wo, minlength=NPT * 2).reshape(NPT, 2)
                wsum[(leg, st, cat)][..., 1] += np.bincount(idx, weights=wn, minlength=NPT * 2).reshape(NPT, 2)
        print(f'{leg}: {n:,} entries, {time.time() - t0:.0f} s', flush=True)
    check['entries'][leg] = n; check['mismatch_old'][leg] = mis_o; check['mismatch_new'][leg] = mis_n
print(check)

seed: 4,000,000 entries, 3 s


seed: 8,000,000 entries, 6 s


seed: 12,000,000 entries, 10 s


seed: 16,000,000 entries, 13 s


seed: 20,000,000 entries, 16 s


...

unseed: 148,000,000 entries, 120 s


unseed: 152,000,000 entries, 123 s


unseed: 156,000,000 entries, 127 s


unseed: 159,861,517 entries, 130 s


{'entries': {'seed': 159861517, 'unseed': 159861517}, 'mismatch_old': {'seed': 0, 'unseed': 0}, 'mismatch_new': {'seed': 0, 'unseed': 0}, 'selected': {}}


## [E] Validate
Assertions: full trees read; zero lookup mismatches; weighted yields rebuilt from cell counts equal the direct sums of stored weights (float32 storage → relative tolerance 1e-5).

In [5]:
# [E] Validate
assert all(v == 159_861_517 for v in check['entries'].values()), check['entries']
assert all(v == 0 for v in check['mismatch_old'].values()), check['mismatch_old']
assert all(v == 0 for v in check['mismatch_new'].values()), check['mismatch_new']
worst = 0.0
for key in cfg:
    for j, mk in enumerate(('old', 'new')):
        rebuilt = (counts[key] * maps[mk].ravel()[None, None, :]).sum(-1)
        np.testing.assert_allclose(rebuilt, wsum[key][..., j], rtol=1e-5, atol=1e-6)
        nz = wsum[key][..., j] != 0
        worst = max(worst, float(np.max(np.abs(rebuilt[nz] / wsum[key][..., j][nz] - 1))) if nz.any() else 0)
    npt = len(cfg[key]['pt']) - 1
    assert counts[key][npt:].sum() == 0
    check['selected']['/'.join(key)] = int(counts[key].sum())
check['max_rel_diff_rebuilt_vs_stored'] = worst
print('lookup mismatches: 0 (old and new); max relative diff rebuilt vs stored:', worst)
print('selected probes, normal set:', {k: v for k, v in check['selected'].items() if '/normal/' in k})

lookup mismatches: 0 (old and new); max relative diff rebuilt vs stored: 1.3907472951046884e-08
selected probes, normal set: {'seed/normal/EB_R9_0p50to0p55': 610203, 'seed/normal/EB_R9_0p55to0p85': 10623354, 'seed/normal/EB_R9gt0p85': 27285255, 'seed/normal/EE_R9_0p90to0p93': 920092, 'seed/normal/EE_R9gt0p93': 8643484, 'unseed/normal/EB_R9_0p50to0p55': 773578, 'unseed/normal/EB_R9_0p55to0p85': 12086789, 'unseed/normal/EB_R9gt0p85': 29556173, 'unseed/normal/EE_R9_0p90to0p93': 978124, 'unseed/normal/EE_R9gt0p93': 9468312}


## [F] Save
`np.savez_compressed` stores counts, stored-weight sums, both maps and edges; the validation numbers go to JSON.

In [6]:
# [F] Save
arrs = {'r9_edges': R9E, 'eta_edges': ETAE, 'map_old': maps['old'], 'map_new': maps['new']}
for key in cfg:
    tag = '__'.join(key)
    arrs['counts__' + tag] = counts[key]
    arrs['wsum__' + tag] = wsum[key]
    arrs['pt__' + tag] = cfg[key]['pt']
np.savez_compressed(OUT / 'cell_counts.npz', **arrs)
check['mass_window'] = MASS_WIN
check['maps'] = {k: str(v) for k, v in MAPS.items()}
(OUT / 'scan_validation.json').write_text(json.dumps(check, indent=1))
print(OUT / 'cell_counts.npz'); print(OUT / 'scan_validation.json')

/home/santanu/Desktop/HLT_2024_lowmass_final_with_NR_eta_r9_rw/etaR9_map_study/data/cell_counts.npz
/home/santanu/Desktop/HLT_2024_lowmass_final_with_NR_eta_r9_rw/etaR9_map_study/data/scan_validation.json
